In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
from openai import OpenAI
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key = os.getenv("GROQ_API_KEY")
)

In [14]:
from google import genai
genai_client = genai.Client(
    api_key = os.environ.get("GENAI_API_KEY")
)

In [15]:
from sqlitesearch import TextSearchIndex
index = TextSearchIndex(
    text_fields= ['question','section','answer'],
    keyword_fields=['course'],
    db_path="database/FAQ.db"
)
index.search("just discovered the course, can i join now?")

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'e0a95572a6',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How do I join Slack if the invite email didn’t arrive?',
  'answer': 'Go to DataTalks.Club, request a Slack invite, or use the manual request form (processed daily). After joining, browse channels and join **#course-ml-zoomcamp**.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you

In [16]:
def search(query):
    boost_dict = {'question':3.0, 'section':.5}
    filter_dict = {'course': 'llm-zoomcamp'}
    
    return index.search(
        query,
        num_results = 5,
        boost_dict=boost_dict,
        filter_dict= filter_dict
    )

In [17]:
import json
search_tool = {
    'type' : 'function',
    'function' : {
        'name' : 'search',
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters" : {
            "type": "object",
            "properties": {
                "query": {
                    "type" : "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required" : ["query"],
            "additionalProperties" : False
        }
    }
}

In [18]:
def make_call(call):
    args = json.loads(call.arguments)
    
    if call.name == "search":
        result = search(**args)
        
    result_json = json.dumps(result)
    
    return {
        "type" : "function_call_output",
        "call_id" : call.call_id,
    }

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [20]:
model = os.getenv("groq_models").split(",")[0]

In [21]:
messages = [
    {'role' : 'system', 'content' : instructions},
    {'role' : 'user', 'content' : question},
]

response = groq_client.chat.completions.create(
    model = model,
    messages = messages,
    tools=[search_tool],
    tool_choice = 'auto',
    max_completion_tokens=1000
)

response_message = response.choices[0].message
messages.append(response_message)

if response_message.tool_calls:
    for tool_call in response_message.tool_calls:
        print("function_call:", tool_call.function.name, tool_call.function.arguments)
        
        args = json.loads(tool_call.function.arguments)
        result = search(**args)
        
        messages.append({
            "role" : "tool",
            "tool_call_id" : tool_call.id,
            "content" : json.dumps(result)
        })
        
    final_response = groq_client.chat.completions.create(
        model = model,
        messages = messages,
        tools=[search_tool]
    )
    print("ASSISTANT:")
    print(final_response.choices[0].message.content)

else:
    print("ASSISTANT:")
    response_message.content


function_call: search {"query":"join course deadline"}
function_call: search {"query":"course enrollment process"}
ASSISTANT:
Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. 

Are there any other areas you'd like to explore or have questions about?


In [22]:
response.choices[0].message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='d0qea7b3v', function=Function(arguments='{"query":"join course deadline"}', name='search'), type='function'), ChatCompletionMessageFunctionToolCall(id='fsj30s562', function=Function(arguments='{"query":"course enrollment process"}', name='search'), type='function')])